# ML-1M data split

Disjoint global timestamp split: **80% train / 10% validation / 10% test**.
Validation and test windows keep only warm users/items and users with both positive and negative feedback.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
from preprocess import train_val_test_split, prepare_splitted_data

PROJECT_PATH = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_PATH, 'data', 'ml-1m')
os.makedirs(DATA_PATH, exist_ok=True)

RELEVANCE_COL = 'rating'
RELEVANCE_THRESHOLD = 4

In [ ]:
# Set ML1M_RATINGS_PATH env var or place ratings.dat under negbt/data/
RAW_PATH = os.environ.get(
    'ML1M_RATINGS_PATH',
    os.path.join(PROJECT_PATH, 'data', 'ratings.dat'),
)

data = pd.read_csv(
    RAW_PATH,
    sep='::',
    engine='python',
    names=['user_id', 'item_id', 'rating', 'timestamp'],
)
data.head()

In [ ]:
print('duplicate user-item pairs:', data.duplicated(['user_id', 'item_id']).sum())
print('negative feedback rate:', np.mean(data[RELEVANCE_COL] < RELEVANCE_THRESHOLD))
print('median interactions per user:', data.groupby('user_id').size().median())

In [ ]:
train, val, test = train_val_test_split(
    data,
    RELEVANCE_THRESHOLD,
    RELEVANCE_COL,
    train_quantile=0.8,
    val_quantile=0.9,
)

In [ ]:
print('train users:', train.user_id.nunique())
print('val users:', val.user_id.nunique())
print('test users:', test.user_id.nunique())
print('val neg rate:', np.mean(val[RELEVANCE_COL] < RELEVANCE_THRESHOLD))
print('test neg rate:', np.mean(test[RELEVANCE_COL] < RELEVANCE_THRESHOLD))

In [ ]:
train.to_parquet(os.path.join(DATA_PATH, 'train.parquet'), index=False)
val.to_parquet(os.path.join(DATA_PATH, 'validation.parquet'), index=False)
test.to_parquet(os.path.join(DATA_PATH, 'test.parquet'), index=False)
print('saved to', DATA_PATH)

In [ ]:
# Sanity check: neighbour-pair target extraction + leak checks
(
    train_p,
    validation_p,
    test_p,
    last_pos_item_test,
    last_pos_item_val,
    last_neg_item_test,
    last_neg_item_val,
) = prepare_splitted_data(
    DATA_PATH,
    relevance_col=RELEVANCE_COL,
    relevance_threshold=RELEVANCE_THRESHOLD,
    verify=True,
)

print('val users with neighbour pair:', last_pos_item_val.user_id.nunique())
print('test users with neighbour pair:', last_pos_item_test.user_id.nunique())